# Auditoría territorial 2020–2025
## Validación de comparabilidad geográfica municipal

### Objetivo

Antes de construir la base maestra del modelo econométrico se revisa la comparabilidad territorial de los municipios entre DENUE 2020 y DENUE 2025.

Durante el periodo de análisis se crearon nuevas áreas municipales a partir del territorio de municipios previamente existentes.

Por esta razón, que una clave `CVEGEO` esté presente tanto en 2020 como en 2025 no garantiza necesariamente que represente exactamente la misma extensión territorial.

Para evitar atribuir a crecimiento comercial cambios que en realidad puedan deberse a modificaciones territoriales, se adopta un criterio conservador:

- identificar los municipios de origen afectados por la creación de nuevos municipios;
- verificar cuáles pertenecen a la muestra candidata del proyecto;
- excluir dichas unidades de la muestra utilizada para construir el modelo.

No se reconstruirán artificialmente fronteras municipales ni se reasignarán establecimientos entre territorios.

In [3]:
# ============================================================
# Reconstrucción de la muestra candidata con cobertura completa, no se guardo por eso tengo que generarla
# ============================================================

archivo_muestra_ilmm = (
    PROCESSED_DIR
    / "muestra_candidata_denue_conapo_censo_ilmm.csv"
)

archivo_coneval = (
    PROCESSED_DIR
    / "coneval_2020_municipal.csv"
)

# Cargar muestra previa e indicador CONEVAL
muestra_ilmm = pd.read_csv(
    archivo_muestra_ilmm,
    dtype={"CVEGEO": "string"}
)

coneval_2020 = pd.read_csv(
    archivo_coneval,
    dtype={"CVEGEO": "string"}
)

print(
    "Muestra antes de CONEVAL:",
    f"{len(muestra_ilmm):,}"
)

print(
    "Municipios CONEVAL:",
    f"{len(coneval_2020):,}"
)

Muestra antes de CONEVAL: 2,458
Municipios CONEVAL: 2,469


In [4]:
# Incorporar únicamente POBREZA_2020 para evaluar cobertura

auditoria_coneval = (
    muestra_ilmm[["CVEGEO"]]
    .merge(
        coneval_2020[
            [
                "CVEGEO",
                "POBREZA_2020"
            ]
        ],
        on="CVEGEO",
        how="left",
        validate="one_to_one"
    )
)

print(
    "Municipios evaluados:",
    f"{len(auditoria_coneval):,}"
)

print(
    "POBREZA_2020 disponible:",
    f"{auditoria_coneval['POBREZA_2020'].notna().sum():,}"
)

print(
    "POBREZA_2020 faltante:",
    f"{auditoria_coneval['POBREZA_2020'].isna().sum():,}"
)

display(
    auditoria_coneval[
        auditoria_coneval["POBREZA_2020"].isna()
    ]
)

Municipios evaluados: 2,458
POBREZA_2020 disponible: 2,457
POBREZA_2020 faltante: 1


,CVEGEO,POBREZA_2020
2069,29048,NaN


In [5]:
muestra_candidata_cobertura_completa = (
    auditoria_coneval[
        auditoria_coneval["POBREZA_2020"].notna()
    ][["CVEGEO"]]
    .copy()
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Municipios con cobertura completa:",
    f"{len(muestra_candidata_cobertura_completa):,}"
)

print(
    "CVEGEO únicos:",
    muestra_candidata_cobertura_completa[
        "CVEGEO"
    ].nunique()
)

print(
    "CVEGEO duplicados:",
    muestra_candidata_cobertura_completa[
        "CVEGEO"
    ].duplicated().sum()
)

Municipios con cobertura completa: 2,457
CVEGEO únicos: 2457
CVEGEO duplicados: 0


In [6]:
archivo_muestra = (
    PROCESSED_DIR
    / "muestra_candidata_cobertura_completa.csv"
)

muestra_candidata_cobertura_completa.to_csv(
    archivo_muestra,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Muestra candidata con cobertura completa "
    "guardada correctamente."
)

print(archivo_muestra)
print("¿Existe el archivo?:", archivo_muestra.exists())

Muestra candidata con cobertura completa guardada correctamente.
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\processed\muestra_candidata_cobertura_completa.csv
¿Existe el archivo?: True


In [ ]:



archivo_muestra = (
    PROCESSED_DIR
    / "muestra_candidata_cobertura_completa.csv"
)

muestra_candidata = pd.read_csv(
    archivo_muestra,
    dtype={"CVEGEO": "string"}
)

print(
    "Municipios con cobertura completa antes "
    "de auditoría territorial:",
    f"{len(muestra_candidata):,}"
)

print(
    "CVEGEO únicos:",
    muestra_candidata["CVEGEO"].nunique()
)

Municipios con cobertura completa antes de auditoría territorial: 2,457
CVEGEO únicos: 2457


In [8]:
"municipios_origen_afectados" in globals()

False

### Municipios de origen afectados por cambios territoriales 2020–2025

Durante el periodo de análisis se crearon nuevos municipios a partir del territorio de municipios previamente existentes.

Para evitar comparar unidades territoriales que no representan exactamente la misma extensión geográfica entre 2020 y 2025, se identifican los municipios de origen afectados.

La estrategia adoptada es conservadora: estos municipios serán marcados para revisión y, posteriormente, se evaluará su exclusión de la muestra utilizada para construir la variable de crecimiento comercial.

In [10]:
municipios_origen_afectados = pd.DataFrame(
    [
        # Baja California
        ["02001", "Baja California", "Ensenada", "San Quintín / San Felipe"],
        ["02002", "Baja California", "Mexicali", "San Felipe"],

        # Campeche
        ["04001", "Campeche", "Calkiní", "Dzitbalché"],
        ["04002", "Campeche", "Campeche", "Seybaplaya"],
        ["04004", "Campeche", "Champotón", "Seybaplaya"],

        # Chiapas
        ["07080", "Chiapas", "Siltepec", "Honduras de la Sierra"],

        # Guerrero
        ["12012", "Guerrero", "Ayutla de los Libres", "Ñuu Savi"],
        ["12023", "Guerrero", "Cuajinicuilapa", "San Nicolás"],
        ["12041", "Guerrero", "Malinaltepec", "Santa Cruz del Rincón"],
        ["12053", "Guerrero", "San Marcos", "Las Vigas"],

        # Morelos
        ["17022", "Morelos", "Tetela del Volcán", "Hueyapan"],

        # San Luis Potosí
        ["24028", "San Luis Potosí", "San Luis Potosí", "Villa de Pozos"],

        # Sinaloa
        ["25006", "Sinaloa", "Culiacán", "Eldorado"],
        ["25001", "Sinaloa", "Ahome", "Juan José Ríos"],
        ["25010", "Sinaloa", "El Fuerte", "Juan José Ríos"],
        ["25011", "Sinaloa", "Guasave", "Juan José Ríos"],
        ["25017", "Sinaloa", "Sinaloa", "Juan José Ríos"],
    ],
    columns=[
        "CVEGEO",
        "entidad",
        "municipio_origen",
        "municipio_nuevo_asociado"
    ]
)

print(
    "Municipios de origen territorialmente afectados:",
    len(municipios_origen_afectados)
)

display(municipios_origen_afectados)

Municipios de origen territorialmente afectados: 17


,CVEGEO,entidad,municipio_origen,municipio_nuevo_asociado
0,02001,Baja California,Ensenada,San Quintín / San Felipe
1,02002,Baja California,Mexicali,San Felipe
2,04001,Campeche,Calkiní,Dzitbalché
3,04002,Campeche,Campeche,Seybaplaya
4,04004,Campeche,Champotón,Seybaplaya
5,07080,Chiapas,Siltepec,Honduras de la Sierra
6,12012,Guerrero,Ayutla de los Libres,Ñuu Savi
7,12023,Guerrero,Cuajinicuilapa,San Nicolás
8,12041,Guerrero,Malinaltepec,Santa Cruz del Rincón
9,12053,Guerrero,San Marcos,Las Vigas


In [11]:
"municipios_origen_afectados" in globals()

True

In [12]:
claves_muestra = set(
    muestra_candidata["CVEGEO"]
)

municipios_afectados_en_muestra = (
    municipios_origen_afectados[
        municipios_origen_afectados["CVEGEO"]
        .isin(claves_muestra)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Municipios territorialmente afectados "
    "presentes en la muestra:",
    len(municipios_afectados_en_muestra)
)

display(municipios_afectados_en_muestra)

Municipios territorialmente afectados presentes en la muestra: 17


,CVEGEO,entidad,municipio_origen,municipio_nuevo_asociado
0,02001,Baja California,Ensenada,San Quintín / San Felipe
1,02002,Baja California,Mexicali,San Felipe
2,04001,Campeche,Calkiní,Dzitbalché
3,04002,Campeche,Campeche,Seybaplaya
4,04004,Campeche,Champotón,Seybaplaya
5,07080,Chiapas,Siltepec,Honduras de la Sierra
6,12012,Guerrero,Ayutla de los Libres,Ñuu Savi
7,12023,Guerrero,Cuajinicuilapa,San Nicolás
8,12041,Guerrero,Malinaltepec,Santa Cruz del Rincón
9,12053,Guerrero,San Marcos,Las Vigas


### Validación de los municipios territorialmente afectados

Antes de excluir los municipios identificados como territorialmente modificados, se verifica su presencia en DENUE 2020 y DENUE 2025.

Esta revisión permite documentar que las unidades existen en ambos cortes, pero que su comparabilidad temporal puede estar afectada por modificaciones en sus límites territoriales.

La exclusión no se debe a falta de información, sino a un criterio de comparabilidad geográfica.

In [13]:
archivo_denue_2020 = (
    PROCESSED_DIR
    / "denue_2020_municipal.csv"
)

archivo_denue_2025 = (
    PROCESSED_DIR
    / "denue_2025_municipal.csv"
)

denue_2020 = pd.read_csv(
    archivo_denue_2020,
    dtype={"CVEGEO": "string"}
)

denue_2025 = pd.read_csv(
    archivo_denue_2025,
    dtype={"CVEGEO": "string"}
)

detalle_afectados = (
    municipios_afectados_en_muestra
    .merge(
        denue_2020[
            ["CVEGEO", "EST_RETAIL_2020"]
        ],
        on="CVEGEO",
        how="left",
        validate="one_to_one"
    )
    .merge(
        denue_2025[
            ["CVEGEO", "EST_RETAIL_2025"]
        ],
        on="CVEGEO",
        how="left",
        validate="one_to_one"
    )
)

display(detalle_afectados)

,CVEGEO,entidad,municipio_origen,municipio_nuevo_asociado,EST_RETAIL_2020,EST_RETAIL_2025
0,02001,Baja California,Ensenada,San Quintín / San Felipe,7610,6767
1,02002,Baja California,Mexicali,San Felipe,11185,12257
2,04001,Campeche,Calkiní,Dzitbalché,1028,792
3,04002,Campeche,Campeche,Seybaplaya,5748,5946
4,04004,Campeche,Champotón,Seybaplaya,1842,1662
5,07080,Chiapas,Siltepec,Honduras de la Sierra,309,304
6,12012,Guerrero,Ayutla de los Libres,Ñuu Savi,774,1116
7,12023,Guerrero,Cuajinicuilapa,San Nicolás,557,560
8,12041,Guerrero,Malinaltepec,Santa Cruz del Rincón,55,67
9,12053,Guerrero,San Marcos,Las Vigas,999,860


In [14]:
print(
    "EST_RETAIL_2020 faltantes:",
    detalle_afectados["EST_RETAIL_2020"].isna().sum()
)

print(
    "EST_RETAIL_2025 faltantes:",
    detalle_afectados["EST_RETAIL_2025"].isna().sum()
)

EST_RETAIL_2020 faltantes: 0
EST_RETAIL_2025 faltantes: 0


### Construcción de la muestra territorialmente comparable

Los municipios de origen afectados por modificaciones territoriales entre 2020 y 2025 se excluyen de la muestra candidata.

Esta decisión busca evitar que los cambios en el número o densidad de establecimientos sean atribuibles parcialmente a modificaciones en los límites municipales y no al comportamiento económico del territorio.

No se reconstruyen artificialmente las fronteras ni se redistribuyen establecimientos o población.

La exclusión se realiza exclusivamente mediante `CVEGEO`.

In [15]:
claves_afectadas = set(
    municipios_afectados_en_muestra["CVEGEO"]
)

muestra_territorial_comparable = (
    muestra_candidata[
        ~muestra_candidata["CVEGEO"]
        .isin(claves_afectadas)
    ]
    .copy()
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Muestra antes de auditoría territorial:",
    f"{len(muestra_candidata):,}"
)

print(
    "Municipios excluidos:",
    f"{len(claves_afectadas):,}"
)

print(
    "Muestra territorialmente comparable:",
    f"{len(muestra_territorial_comparable):,}"
)

Muestra antes de auditoría territorial: 2,457
Municipios excluidos: 17
Muestra territorialmente comparable: 2,440


In [16]:
print("CONTROL DE CALIDAD — MUESTRA TERRITORIAL")
print("=" * 55)

print(
    "Número de municipios:",
    f"{len(muestra_territorial_comparable):,}"
)

print(
    "CVEGEO únicos:",
    f"{muestra_territorial_comparable['CVEGEO'].nunique():,}"
)

print(
    "CVEGEO duplicados:",
    muestra_territorial_comparable["CVEGEO"]
    .duplicated()
    .sum()
)

print(
    "Municipios afectados todavía presentes:",
    muestra_territorial_comparable["CVEGEO"]
    .isin(claves_afectadas)
    .sum()
)

CONTROL DE CALIDAD — MUESTRA TERRITORIAL
Número de municipios: 2,440
CVEGEO únicos: 2,440
CVEGEO duplicados: 0
Municipios afectados todavía presentes: 0


In [17]:
assert len(muestra_territorial_comparable) == 2440, \
    "El tamaño esperado de la muestra no coincide."

assert (
    muestra_territorial_comparable["CVEGEO"].nunique()
    == 2440
), "Existen problemas de unicidad."

assert (
    muestra_territorial_comparable["CVEGEO"]
    .duplicated()
    .sum()
    == 0
), "Existen CVEGEO duplicados."

assert (
    muestra_territorial_comparable["CVEGEO"]
    .isin(claves_afectadas)
    .sum()
    == 0
), "Todavía existen municipios territorialmente afectados."

print(
    "Todas las validaciones territoriales "
    "fueron superadas correctamente."
)

Todas las validaciones territoriales fueron superadas correctamente.


In [18]:
##### 9. Guardar la muestra comparable

archivo_muestra_territorial = (
    PROCESSED_DIR
    / "muestra_territorialmente_comparable.csv"
)

muestra_territorial_comparable.to_csv(
    archivo_muestra_territorial,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Muestra territorialmente comparable "
    "guardada correctamente."
)

print(archivo_muestra_territorial)

print(
    "¿Existe el archivo?:",
    archivo_muestra_territorial.exists()
)

Muestra territorialmente comparable guardada correctamente.
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\processed\muestra_territorialmente_comparable.csv
¿Existe el archivo?: True


## Conclusión de la auditoría territorial 2020–2025

La muestra candidata con cobertura completa contenía 2,457 municipios.

La revisión territorial identificó 17 municipios de origen cuya delimitación fue afectada por la creación de nuevos municipios durante el periodo de análisis.

Aunque estos municipios cuentan con información en DENUE 2020 y DENUE 2025, se decidió excluirlos mediante un criterio conservador de comparabilidad geográfica.

Esta decisión busca evitar que cambios en la densidad comercial sean consecuencia parcial de modificaciones territoriales y no del comportamiento económico del municipio.

Resultados:

- Muestra con cobertura completa antes de la auditoría: 2,457 municipios.
- Municipios de origen territorialmente afectados: 17.
- Municipios excluidos: 17.
- Muestra territorialmente comparable: 2,440 municipios.
- CVEGEO duplicados: 0.
- Municipios territorialmente afectados remanentes: 0.

La muestra de 2,440 municipios será utilizada para construir la base maestra del proyecto.

Su tamaño definitivo será confirmado después de integrar Y, X1, X2, X3 y X4 y realizar los controles finales de calidad.